# 06 — Screening Training Data by Projection

**Paper Section 6:** Before finetuning, you can predict which *individual samples*
will push the model along a persona direction — by projecting each sample's
response activation onto the persona vector. High-projection samples are likely
to reinforce the trait if trained on.

In this notebook you:

1. Load a mixed dataset — clean samples merged with flawed ones.
2. Score every sample's response activation projection onto the `evil` vector.
3. Rank samples, inspect the top / bottom, and compute AUROC against the
   ground-truth clean/flawed label.


## Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

from persona_lib import ActivationModel, ChatPrompt, load_extraction, score_samples
from persona_lib.data import read_jsonl

TRAIT = 'evil'
MODEL = "Qwen/Qwen2.5-7B-Instruct"
CLEAN = "data/finetune/medical_normal.jsonl"
FLAWED = "data/finetune/medical_II.jsonl"

ext = load_extraction(TRAIT)
L, v = ext.best_layer, ext.vectors_by_layer[ext.best_layer]


## Build a mixed dataset with ground-truth labels


In [ ]:
clean = read_jsonl(CLEAN)
flawed = read_jsonl(FLAWED)
for r in clean: r['label'] = 0
for r in flawed: r['label'] = 1

import random
rng = random.Random(0)
mixed = clean + flawed
rng.shuffle(mixed)
print(f"mixed set: {len(mixed)} ({sum(r['label'] for r in mixed)} flawed)")


## Step 1 — Score every sample

`score_samples` collects each sample's mean residual-stream activation over its
assistant response, then projects onto the unit persona vector.


In [ ]:
model = ActivationModel(MODEL)
samples = [ChatPrompt(user=r['user'], assistant=r['assistant']) for r in mixed]
scores = score_samples(model, samples, v, layer=L)
for r, s in zip(mixed, scores):
    r['projection'] = float(s)
print(f"projection range: [{scores.min():.2f}, {scores.max():.2f}]")


## TODO 1 — AUROC of projection for flagging flawed samples


In [ ]:
labels = np.array([r['label'] for r in mixed])
# TODO: compute the AUROC of `scores` as a predictor of `labels`.
auc = ...  # YOUR CODE HERE
print(f"AUROC (projection → flawed): {auc:.3f}")


## TODO 2 — Plot the ROC curve


In [ ]:
# TODO: use sklearn.metrics.roc_curve to plot TPR vs FPR.
...  # YOUR CODE HERE


## Qualitative inspection — top and bottom of the ranking


In [ ]:
ranked = sorted(mixed, key=lambda r: -r['projection'])
print("=== TOP 3 (most aligned with evil) ===")
for r in ranked[:3]:
    print(f"  [label={r['label']}] proj={r['projection']:.2f}")
    print(f"  user: {r['user'][:100]}")
    print(f"  asst: {r['assistant'][:150]}")
    print()
print("=== BOTTOM 3 ===")
for r in ranked[-3:]:
    print(f"  [label={r['label']}] proj={r['projection']:.2f}")
    print(f"  user: {r['user'][:100]}")
    print(f"  asst: {r['assistant'][:150]}")
    print()


## Analysis

1. What AUROC did you get? The paper shows projection screening catches samples
   that evade LLM-based data filters.
2. Look at any false positives (clean samples with high projection). What common
   property makes them "look evil" to the vector? Is this a feature or a bug?
3. What's the capstone-worthy follow-up? E.g., screening a real public dataset
   slice (Alpaca, WildChat) — what does the top-100 look like?
